# Gemma 3 Pig Latin Fine-Tuning with Open-RL

Pig Latin is small enough that the notebook can stay simple.

We load a few word pairs, sample the untouched model through Open-RL, run a short LoRA loop, compare a few held-out words, and then try our own words against the trained adapter.


In [1]:
import json
import os
import random
import re
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tinker
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tinker import types

os.environ.setdefault("TINKER_API_KEY", "tml-dummy-key")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

%matplotlib inline
plt.rcParams["figure.dpi"] = 120


/private/tmp/open-rl-piglatin/client/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Before You Run It

Sync the client env with `cd client && uv sync`, then start `make run-pig-latin-gemma-server` in another terminal. The notebook talks to `http://127.0.0.1:9002`.


In [2]:
DEFAULT_BASE_URL = os.getenv("TINKER_BASE_URL") or os.getenv("OPEN_RL_BASE_URL") or "http://127.0.0.1:9002"
PAIRS_PATH = Path("piglatin_data.json")
SYSTEM_PROMPT = "Translate the English text into Pig Latin. Reply with only the Pig Latin translation."
EVAL_MAX_TOKENS = 32


@dataclass
class Config:
    base_model: str = "google/gemma-3-1b-it"
    tokenizer_model: str = "google/gemma-3-1b-it"
    steps: int = 25
    batch_size: int = 16
    lora_rank: int = 16
    learning_rate: float = 1e-4
    server_command: str = "make run-pig-latin-gemma-server"
    base_url: str = DEFAULT_BASE_URL
    eval_every: int = 12
    train_limit: int = 320
    eval_limit: int = 20
    seed: int = 64
    metrics_path: str = "artifacts/piglatin_notebook_metrics.jsonl"
    plot_path: str = "artifacts/piglatin_notebook_results.png"
    run_label: str = "Gemma 3 Pig Latin SFT"


config = Config()
metrics_path = Path(config.metrics_path)
plot_path = Path(config.plot_path)

config_table = pd.DataFrame.from_dict(asdict(config), orient="index", columns=["value"])
display(config_table)


,value
base_model,google/gemma-3-1b-it
tokenizer_model,google/gemma-3-1b-it
steps,25
batch_size,16
lora_rank,16
learning_rate,0.0001
server_command,make run-pig-latin-gemma-server
base_url,http://127.0.0.1:9002
eval_every,12
train_limit,320


## Data

Load the word pairs and preview a few rows in the same block. The point here is just to see the source word and target Pig Latin translation before we start sampling or training.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_model)

pair_data = json.loads(PAIRS_PATH.read_text())
train_pairs = list(pair_data["train"])
eval_pairs = list(pair_data["eval"])
random.Random(config.seed).shuffle(train_pairs)

print(f"Training pairs: {len(train_pairs):,}")
print(f"Eval pairs: {len(eval_pairs):,}")


PAIR_PREVIEW_IDS = [0, 1, 2, 3, 4]
PLAYGROUND_WORDS = ["banana", "machine", "coding"]

pair_preview_df = pd.DataFrame(
    [
        {"source": train_pairs[i][0], "target": train_pairs[i][1]}
        for i in PAIR_PREVIEW_IDS
    ]
)
print("A few raw training pairs")
display(pair_preview_df)

print("The eval slice is chosen later from the built eval examples.")


Training pairs: 319
Eval pairs: 80
A few raw training pairs


,source,target
0,kitchen,itchen-kay
1,attic,attic-way
2,butterfly,utterfly-bay
3,turtle,urtle-tay
4,loud,oud-lay


The eval slice is chosen later from the built eval examples.


## Backend and Training Loop

The next cells follow the README pattern: connect to Open-RL through Tinker, build examples, sample the untouched model, then run the short loop of batch -> `forward_backward_async(...)` -> `optim_step_async(...)` -> eval.


In [4]:
async def get_server_model(service_client):
    try:
        capabilities = await service_client.get_server_capabilities_async()
    except Exception as exc:
        message = (
            f"Open-RL server at {config.base_url} is not reachable. "
            f"Start it with `{config.server_command}`."
        )
        raise RuntimeError(message) from exc

    for model in capabilities.supported_models:
        model_name = getattr(model, "model_name", None)
        if model_name:
            return model_name

    return None


service_client = tinker.ServiceClient(
    api_key=os.getenv("TINKER_API_KEY", "tml-dummy-key"),
    base_url=config.base_url,
)

server_model = await get_server_model(service_client)
if server_model is None:
    raise RuntimeError(
        f"Server at {config.base_url} is reachable, but no model is loaded. "
        f"Start it with `{config.server_command}`."
    )

print(f"Connected to {config.base_url}")
print(f"Server model: {server_model}")

training_client = await service_client.create_lora_training_client_async(
    base_model=config.base_model,
    rank=config.lora_rank,
    train_mlp=True,
    train_attn=True,
    train_unembed=False,
)
print(f"Created LoRA training client | base_model={config.base_model} | rank={config.lora_rank}")


Connected to http://127.0.0.1:9002
Server model: google/gemma-3-1b-it
Created LoRA training client | base_model=google/gemma-3-1b-it | rank=16


In [5]:
def build_messages(source, target):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": source},
        {"role": "assistant", "content": target},
    ]


def make_datum(token_ids, weights):
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=token_ids[:-1]),
        loss_fn_inputs={
            "weights": weights[1:],
            "target_tokens": token_ids[1:],
        },
    )


def build_example(pair):
    source, target = pair
    messages = build_messages(source, target)

    # Build both the prompt-only text used for sampling and the full prompt+answer text used for training.
    prompt_text = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    full_token_ids = tokenizer.encode(full_text, add_special_tokens=False)
    if len(full_token_ids) <= len(prompt_token_ids):
        return None

    target_token_count = len(full_token_ids) - len(prompt_token_ids)
    weights = [0] * len(prompt_token_ids) + [1] * target_token_count

    return {
        "messages": messages,
        "source": source,
        "target": target,
        "prompt_tokens": prompt_token_ids,
        "active_tokens": target_token_count,
        "datum": make_datum(full_token_ids, weights),
    }


train_examples = [ex for pair in train_pairs[:config.train_limit] if (ex := build_example(pair)) is not None]
eval_examples = [ex for pair in eval_pairs[:config.eval_limit] if (ex := build_example(pair)) is not None]

if not train_examples:
    raise RuntimeError("No training examples were built.")

batch_size = min(config.batch_size, len(train_examples))
print(f"Usable training examples: {len(train_examples):,}")
print(f"Usable eval examples: {len(eval_examples):,}")
print(f"Batch size: {batch_size}")

showcase_eval_examples = eval_examples[:5]
baseline_examples = showcase_eval_examples[:2]
comparison_examples = showcase_eval_examples[2:]

eval_selection_df = pd.DataFrame(
    [
        {
            "group": label,
            "source": example["source"],
            "target": example["target"],
        }
        for label, examples in [
            ("Base model feel", baseline_examples),
            ("Before vs after compare", comparison_examples),
        ]
        for example in examples
    ]
)
print("Eval pairs used later in the notebook")
display(eval_selection_df)

print("One formatted prompt/completion example")
display(pd.DataFrame(eval_examples[0]["messages"]))


Usable training examples: 319
Usable eval examples: 20
Batch size: 16
Eval pairs used later in the notebook


,group,source,target
0,Base model feel,good,ood-gay
1,Base model feel,bad,ad-bay
2,Before vs after compare,great,eat-gray
3,Before vs after compare,terrible,errible-tay
4,Before vs after compare,awesome,awesome-way


One formatted prompt/completion example


,role,content
0,system,Translate the English text into Pig Latin. Rep...
1,user,good
2,assistant,ood-gay


In [6]:
def normalize_translation(text):
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = text.replace("<|im_start|>", " ").replace("<|im_end|>", " ")
    text = re.sub(r"^assistant\s*[:\-]?\s*", "", text.strip(), flags=re.IGNORECASE)
    return " ".join(text.split()).lower()


def create_sampler(alias):
    # Snapshot the current adapter state and reopen it through the sampling API.
    weights_path = training_client.save_weights_for_sampler(name=alias).result().path
    return service_client.create_sampling_client(weights_path)


def decode_first_sequence(result):
    if result.sequences:
        token_ids = result.sequences[0].tokens
    else:
        token_ids = []
    return tokenizer.decode(token_ids, skip_special_tokens=True)


def run_eval(alias, examples):
    sampler = create_sampler(alias)
    pending_requests = []
    for example in examples:
        future = sampler.sample(
            prompt=types.ModelInput.from_ints(tokens=example["prompt_tokens"]),
            num_samples=1,
            sampling_params=types.SamplingParams(max_tokens=EVAL_MAX_TOKENS, temperature=0.0),
        )
        pending_requests.append((example, future))

    rows = []
    exact_match_count = 0
    similarity_sum = 0.0
    for example, future in pending_requests:
        result = future.result()
        raw_prediction = decode_first_sequence(result).strip()
        predicted = normalize_translation(raw_prediction)
        target = normalize_translation(example["target"])
        similarity = SequenceMatcher(None, predicted, target).ratio()
        exact_match = predicted == target
        exact_match_count += float(exact_match)
        similarity_sum += similarity
        rows.append(
            {
                "source": example["source"],
                "target_raw": example["target"],
                "predicted_raw": raw_prediction,
                "exact": exact_match,
                "similarity": similarity,
            }
        )

    details_df = pd.DataFrame(rows)
    row_count = max(1, len(rows))
    return exact_match_count / row_count, similarity_sum / row_count, details_df


def show_eval_examples(title, details_df):
    print(title)
    display(details_df.rename(columns={"target_raw": "target", "predicted_raw": "model_output"}))


def show_eval_comparison(title, before_df, after_df):
    comparison_df = pd.DataFrame(
        {
            "source": before_df["source"],
            "target": before_df["target_raw"],
            "base_output": before_df["predicted_raw"],
            "trained_output": after_df["predicted_raw"],
            "base_exact": before_df["exact"],
            "trained_exact": after_df["exact"],
        }
    )
    print(title)
    display(comparison_df)


In [7]:
before_exact, before_similarity, baseline_df = run_eval("baseline", eval_examples)
baseline_summary_df = pd.DataFrame(
    [
        {"metric": "Exact match", "value": f"{before_exact * 100:.1f}%"},
        {"metric": "Similarity", "value": f"{before_similarity * 100:.1f}%"},
        {"metric": "Eval rows", "value": len(eval_examples)},
    ]
)
display(baseline_summary_df)

_, _, baseline_examples_df = run_eval("baseline_examples", baseline_examples)
show_eval_examples("A couple of concrete untuned examples", baseline_examples_df)

# Keep a separate before-training slice so we can compare the same rows again after training.
_, _, comparison_before_df = run_eval("comparison_before", comparison_examples)


,metric,value
0,Exact match,0.0%
1,Similarity,63.8%
2,Eval rows,20


A couple of concrete untuned examples


,source,target,model_output,exact,similarity
0,good,ood-gay,oodday,False,0.769231
1,bad,ad-bay,aodab,False,0.545455


In [8]:
# Also write metrics to JSONL so the same run can be inspected or plotted later.
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text("", encoding="utf-8")

metrics_log = []


def record_metric(step, phase, loss=None, exact_match=None, similarity=None):
    row = {"step": step, "phase": phase}
    if loss is not None:
        row["loss"] = loss
    if exact_match is not None:
        row["exact_match"] = exact_match
    if similarity is not None:
        row["similarity"] = similarity
    metrics_log.append(row)
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


record_metric(step=0, phase="eval", exact_match=before_exact, similarity=before_similarity)
print(f"Metrics will be written to {metrics_path}")

losses = []
eval_history = [{"step": 0, "exact_match": before_exact, "similarity": before_similarity}]

rng = random.Random(config.seed)
shuffled_indices = list(range(len(train_examples)))
rng.shuffle(shuffled_indices)
position = 0
progress_bar = tqdm(range(1, config.steps + 1), desc="Training", unit="step")

for step in progress_bar:
    # 1. Grab the next batch from a shuffled walk over the dataset.
    if position + batch_size > len(shuffled_indices):
        rng.shuffle(shuffled_indices)
        position = 0

    # Walk through a shuffled index list and reshuffle when we reach the end.
    batch_indices = shuffled_indices[position:position + batch_size]
    position += batch_size
    batch = [train_examples[index] for index in batch_indices]

    # 2. Turn that batch into the Datum objects the backend expects.
    datums = [example["datum"] for example in batch]
    active_token_count = sum(example["active_tokens"] for example in batch)

    # 3. Submit the forward/backward pass and the optimizer step to the backend.
    forward_backward_future = await training_client.forward_backward_async(datums, "cross_entropy")
    optimizer_future = await training_client.optim_step_async(
        types.AdamParams(learning_rate=config.learning_rate)
    )

    forward_backward_result = await forward_backward_future
    await optimizer_future

    # 4. Convert the summed token loss into a per-token number we can track.
    loss_sum = float(forward_backward_result.metrics.get("loss:sum", 0.0))
    loss = loss_sum / max(1, active_token_count)
    losses.append(loss)
    record_metric(step=step, phase="train", loss=loss)
    progress_bar.set_postfix(loss=f"{loss:.4f}")

    # 5. Every so often, snapshot the adapter and run eval through the same sampling path.
    should_evaluate = step % config.eval_every == 0 or step == config.steps
    if should_evaluate:
        exact_match, similarity, _ = run_eval(f"step_{step}", eval_examples)
        eval_history.append({"step": step, "exact_match": exact_match, "similarity": similarity})
        record_metric(step=step, phase="eval", exact_match=exact_match, similarity=similarity)
        progress_bar.write(f"[eval at step {step}] exact={exact_match * 100:.1f}% similarity={similarity * 100:.1f}%")


Metrics will be written to artifacts/piglatin_notebook_metrics.jsonl


Training:  48%|████▊     | 12/25 [01:06<01:21,  6.25s/step, loss=0.8203]

[eval at step 12] exact=25.0% similarity=78.3%


Training:  96%|█████████▌| 24/25 [02:05<00:06,  6.59s/step, loss=0.4200]

[eval at step 24] exact=50.0% similarity=88.8%


Training: 100%|██████████| 25/25 [02:16<00:00,  5.45s/step, loss=0.4432]

[eval at step 25] exact=45.0% similarity=88.1%


## What Changed

Show the summary first, then compare a few held-out words before and after training.


In [9]:
eval_df = pd.DataFrame(eval_history)

initial_loss = losses[0]
final_loss = losses[-1]
loss_drop_percent = (initial_loss - final_loss) / (abs(initial_loss) or 1.0) * 100

summary_rows = [
    {"Metric": "Exact Match (%)", "Before": f"{before_exact * 100:.1f}", "After": f"{eval_history[-1]["exact_match"] * 100:.1f}"},
    {"Metric": "Similarity (%)", "Before": f"{before_similarity * 100:.1f}", "After": f"{eval_history[-1]["similarity"] * 100:.1f}"},
    {"Metric": "Loss", "Before": f"{initial_loss:.4f}", "After": f"{final_loss:.4f}"},
    {"Metric": "Loss Drop (%)", "Before": "-", "After": f"{loss_drop_percent:.1f}"},
]
summary_df = pd.DataFrame(summary_rows).set_index("Metric")
display(summary_df)

print(f"Saved training metrics to {metrics_path}")


,Before,After
Metric,,
Exact Match (%),0.0,45.0
Similarity (%),63.8,88.1
Loss,5.9785,0.4432
Loss Drop (%),-,92.6


Saved training metrics to artifacts/piglatin_notebook_metrics.jsonl


In [10]:
after_exact, after_similarity, after_training_df = run_eval("final", eval_examples)
_, _, comparison_after_df = run_eval("comparison_after", comparison_examples)
show_eval_comparison("Before vs after on a few held-out words", comparison_before_df, comparison_after_df)


Before vs after on a few held-out words


,source,target,base_output,trained_output,base_exact,trained_exact
0,great,eat-gray,اگrietay,ight-gray,False,False
1,terrible,errible-tay,erillibe,e terrible-way,False,False
2,awesome,awesome-way,awesomeway,awesome-way,False,True


## Try Your Own Words


In [11]:
def sample_translation(word, sampler_name="interactive"):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": word},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    prompt_token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    # Sample from the current adapter snapshot, not directly from the base model.
    sampler = create_sampler(sampler_name)
    result = sampler.sample(
        prompt=types.ModelInput.from_ints(tokens=prompt_token_ids),
        num_samples=1,
        sampling_params=types.SamplingParams(max_tokens=EVAL_MAX_TOKENS, temperature=0.0),
    ).result()
    return decode_first_sequence(result).strip()


generated_rows = [
    {"source": word, "generated_translation": sample_translation(word, sampler_name="interactive") or "-- empty --"}
    for word in PLAYGROUND_WORDS
]
display(pd.DataFrame(generated_rows))


,source,generated_translation
0,banana,ana-ban-way
1,machine,achine-may
2,coding,oding-cay
